# MMLU-Pro EDA

phase01 difficulty verification — 다운로드한 MMLU-Pro test split 탐색.

- 데이터: `data/mmlu_pro_test.jsonl`
- 컬럼: `question_id`, `question`, `options`(리스트), `answer`(글자), `answer_index`(정수), `cot_content`, `category`, `src`
- GSM8K와 달리 **객관식(최대 10지선다)** 이라 정답이 글자(A~J)다.

In [ ]:
import json
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 200)
LETTERS = string.ascii_uppercase
DATA_PATH = "data/mmlu_pro_test.jsonl"

## 1. 로드 및 기본 정보

In [ ]:
records = [json.loads(line) for line in open(DATA_PATH, encoding="utf-8")]
df = pd.DataFrame(records)

print("num examples:", len(df))
print("columns     :", list(df.columns))
print("nulls       :\n", df.isna().sum())
print("duplicated questions:", df["question"].duplicated().sum())
df.head(3)

## 2. 파생 피처 추출

- `n_options`: 선택지 개수 (MMLU-Pro는 최대 10개, 일부는 더 적음)
- `q_char_len` / `q_word_len`: 질문 길이
- `opt_char_mean`: 선택지 평균 길이
- `has_cot`: cot_content(정답 풀이) 존재 여부
- `answer_ok`: answer 글자와 answer_index 가 일치하는지 정합성 체크

In [ ]:
df["n_options"]     = df["options"].apply(len)
df["q_char_len"]    = df["question"].str.len()
df["q_word_len"]    = df["question"].str.split().apply(len)
df["opt_char_mean"] = df["options"].apply(lambda opts: np.mean([len(str(o)) for o in opts]))
df["has_cot"]       = df["cot_content"].fillna("").str.strip().str.len() > 0
df["cot_char_len"]  = df["cot_content"].fillna("").str.len()

# 정합성: answer(글자) == LETTERS[answer_index]?
df["answer_ok"] = df.apply(lambda r: r["answer"] == LETTERS[r["answer_index"]], axis=1)
print("answer == LETTERS[answer_index] 모두 일치:", df["answer_ok"].all())

df[["n_options", "q_char_len", "q_word_len", "opt_char_mean", "cot_char_len"]].describe()

## 3. 카테고리(category) 분포

MMLU-Pro는 여러 분야로 구성된다. 앞쪽 N개만 샘플로 쓰는 경우(run_rollouts) 분야 편향이 생길 수 있으니 확인.

In [ ]:
cat_counts = df["category"].value_counts()
print(cat_counts)

plt.figure(figsize=(10, 5))
cat_counts.plot(kind="bar", color="steelblue", edgecolor="white")
plt.title("Category distribution (full test set)")
plt.ylabel("count"); plt.xlabel("category")
plt.tight_layout(); plt.show()

In [ ]:
# run_rollouts 가 앞쪽 NUM_SAMPLES개만 쓰므로, 앞 500개의 카테고리 편향 확인
N = 500
print(f"앞 {N}개 카테고리 분포:")
print(df.head(N)["category"].value_counts())

## 4. 정답 글자 / 선택지 개수 분포

정답 글자가 특정 위치(A 등)에 쏠려 있으면 probe/모델 평가에 편향이 생길 수 있다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ans_counts = df["answer"].value_counts().sort_index()
axes[0].bar(ans_counts.index, ans_counts.values, color="seagreen", edgecolor="white")
axes[0].set_title("Answer letter distribution"); axes[0].set_xlabel("answer")

opt_counts = df["n_options"].value_counts().sort_index()
axes[1].bar(opt_counts.index.astype(str), opt_counts.values, color="darkorange", edgecolor="white")
axes[1].set_title("# options distribution"); axes[1].set_xlabel("n_options")

plt.tight_layout(); plt.show()

## 5. 길이 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
specs = [
    ("q_word_len",    "Question length (words)"),
    ("opt_char_mean", "Mean option length (chars)"),
    ("cot_char_len",  "CoT content length (chars)"),
]
for ax, (col, title) in zip(axes, specs):
    ax.hist(df[col], bins=40, color="steelblue", edgecolor="white")
    ax.axvline(df[col].mean(), color="crimson", ls="--", lw=1.5, label=f"mean={df[col].mean():.1f}")
    ax.set_title(title); ax.set_xlabel(col); ax.set_ylabel("count"); ax.legend()
plt.tight_layout(); plt.show()

print("CoT 풀이가 있는 문제 비율:", f"{df['has_cot'].mean()*100:.1f}%")

## 6. 샘플 살펴보기

In [ ]:
def show(row):
    print("=" * 80)
    print(f"[category={row.category}, n_options={row.n_options}, answer={row.answer}]")
    print("Q:", row.question)
    for i, opt in enumerate(row.options):
        mark = " <-- gold" if LETTERS[i] == row.answer else ""
        print(f"   {LETTERS[i]}) {opt}{mark}")

for _, row in df.head(2).iterrows():
    show(row)
    print()

## 7. 요약 저장 (선택)

In [ ]:
out_cols = ["question_id", "category", "n_options", "answer", "q_word_len", "opt_char_mean", "has_cot"]
df[out_cols].to_csv("data/mmlu_pro_eda_features.csv", index=False)
print("saved -> data/mmlu_pro_eda_features.csv")